# LeetCode 2135 — Count Words Obtained After Adding a Letter bằng Python

Link đề: https://leetcode.com/problems/count-words-obtained-after-adding-a-letter/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, String, Bit Manipulation, Sorting

Skeleton LeetCode:

```python
class Solution:
    def wordCount(self, startWords: List[str], targetWords: List[str]) -> int:
        ...
```

## 1. Mục tiêu học

1. Nhận ra "sắp xếp lại tùy ý" nghĩa là **chỉ tập chữ cái** quan trọng ⇒ biểu diễn chuỗi bằng **tập** (`frozenset`, chuỗi đã sắp xếp, hoặc **bitmask 26 bit**).
2. Đảo chiều câu hỏi: thay vì "từ start thêm chữ nào?", hỏi "**bỏ chữ nào** khỏi target thì ra một start?" — chỉ ≤ 26 khả năng.
3. Thành thạo thao tác bit: `1 << (ord(c) - 97)`, OR để gộp, XOR để bỏ một bit, `in` trên `set` các số nguyên.
4. So sánh 5 cách: vét cạn cặp, chuỗi sắp xếp, `frozenset`, bitmask bỏ-một-chữ, bitmask thêm-một-chữ (chiều thuận).
5. Tránh các bẫy: phải thêm **đúng một** chữ, mỗi target đếm **một lần**, target **trùng nhau** vẫn đếm riêng.

## 2. Đề bài & ví dụ

Cho hai mảng chuỗi `startWords` và `targetWords` (chữ thường; **không chuỗi nào có chữ lặp**).

Với mỗi chuỗi trong `targetWords`, kiểm tra xem có thể chọn **một** chuỗi trong `startWords` và thực hiện **phép biến đổi** để được nó không. Phép biến đổi gồm hai bước:
1. **Thêm** vào cuối một chữ cái thường **chưa có** trong chuỗi.
2. **Sắp xếp lại** các chữ cái theo thứ tự bất kỳ.

Trả về **số chuỗi trong `targetWords`** tạo được. (startWords không bị thay đổi thật, chỉ là kiểm tra.)

**Ví dụ 1:** `startWords = ["ant","act","tack"]`, `targetWords = ["tack","act","acti"]` → `2`.
- `"tack"` ← `"act"` + `'k'`. `"acti"` ← `"act"` + `'i'`.
- `"act"` không được: bắt buộc phải thêm một chữ (không có start nào dài 2 là tập con).

**Ví dụ 2:** `startWords = ["ab","a"]`, `targetWords = ["abc","abcd"]` → `1`.

**Ràng buộc:** `1 ≤ len(startWords), len(targetWords) ≤ 5·10⁴`; mỗi chuỗi dài 1..26; không chữ nào lặp trong một chuỗi.

In [1]:
from typing import List
import random, time, string

examples = [
    (["ant", "act", "tack"], ["tack", "act", "acti"], 2),
    (["ab", "a"], ["abc", "abcd"], 1),
]
for st, tg, exp in examples:
    print("start =", st, "| target =", tg, "-> kỳ vọng", exp)

start = ['ant', 'act', 'tack'] | target = ['tack', 'act', 'acti'] -> kỳ vọng 2
start = ['ab', 'a'] | target = ['abc', 'abcd'] -> kỳ vọng 1


## 3. Kiến thức nền

### 3.1 "Sắp xếp lại tùy ý" ⇒ chỉ tập chữ cái quan trọng

Hai chuỗi (không chữ lặp) là hoán vị của nhau ⇔ cùng **tập** chữ cái ⇔ `sorted(a) == sorted(b)`.

In [2]:
print(sorted("tack") == sorted("actk"), set("tack") == set("kcat"), "".join(sorted("tack")))

True True ackt


### 3.2 `frozenset` — tập bất biến, băm được

`set` không cho vào `set` khác (không băm được); `frozenset` thì được. Phép `fs - {c}` tạo tập mới bỏ `c`.

In [3]:
starts = {frozenset("act"), frozenset("ant")}
t = frozenset("tack")
print([c for c in t if (t - {c}) in starts])

['k']


### 3.3 Bitmask 26 bit

Chữ `c` ứng với bit thứ `ord(c) − 97`. Một tập chữ ↔ một số nguyên `< 2²⁶`.
- Gộp: `mask |= 1 << b`.
- Có chữ `c` không: `mask >> b & 1`.
- Bỏ chữ `c` (đang có): `mask ^ (1 << b)` (hoặc `mask & ~(1 << b)`).

So sánh / băm số nguyên rất nhanh — nhanh hơn băm chuỗi hay frozenset.

In [4]:
def to_mask(w):
    m = 0
    for c in w:
        m |= 1 << (ord(c) - 97)
    return m

m = to_mask("act")
print(bin(m), m, "| có 'c'?", m >> 2 & 1, "| có 'd'?", m >> 3 & 1)
print("bỏ 'a':", bin(m ^ 1), "| = mask('ct')?", (m ^ 1) == to_mask("ct"))
print("thứ tự không quan trọng:", to_mask("tack") == to_mask("kcat"))

0b10000000000000000101 524293 | có 'c'? 1 | có 'd'? 0
bỏ 'a': 0b10000000000000000100 | = mask('ct')? True
thứ tự không quan trọng: True


### 3.4 Độ ưu tiên toán tử với bit (Python khác C/Java!)

Trong Python, thứ tự ưu tiên từ cao xuống thấp: `+ -` → `<< >>` → `&` → `^` → `|` → so sánh (`==`, `<`, `in`, …).
- `1 << ord(c) - 97` = `1 << (ord(c) - 97)` (trừ trước, dịch sau) — đúng ý.
- `x ^ 1 << b in s` = `(x ^ (1 << b)) in s` — đúng ý.
- `x & 1 == 0` = `(x & 1) == 0` — **khác C/Java** (ở C, `==` mạnh hơn `&`). Code Python chạy đúng, nhưng dịch sang C++ sẽ sai ⇒ **luôn đóng ngoặc**.

Một bẫy thật sự hay gặp: kiểm tra bit bằng `m & (1 << b) == 1`. Kết quả của `m & (1 << b)` là `0` hoặc **`1 << b`** (không phải 1!) ⇒ chỉ đúng khi `b = 0`.

In [5]:
x = 6                      # 110
print("x & 1 == 0   ->", x & 1 == 0, "| (x & 1) == 0 ->", (x & 1) == 0, "(Python: giống nhau)")
print(1 << ord("c") - 97, 1 << (ord("c") - 97))
m = to_mask("act")         # bit 0 (a), 2 (c), 19 (t)
print("m & (1 << 2)       =", m & (1 << 2), "-> so với 1:", (m & (1 << 2)) == 1, "(SAI dù 'c' có trong m)")
print("(m >> 2) & 1 == 1  =", (m >> 2) & 1 == 1)

x & 1 == 0   -> True | (x & 1) == 0 -> True (Python: giống nhau)
4 4
m & (1 << 2)       = 4 -> so với 1: False (SAI dù 'c' có trong m)
(m >> 2) & 1 == 1  = True


## 4. Ý tưởng nền tảng / trực giác

Target `t` tạo được từ start `s` ⇔ `set(t)` = `set(s)` ∪ {một chữ mới} ⇔ `len(t) = len(s) + 1` **và** `set(s) ⊂ set(t)`.

Đảo chiều: `t` tạo được ⇔ **tồn tại chữ `c ∈ t`** sao cho `set(t) − {c}` là tập chữ của **một** start nào đó.

Tính tay ví dụ 1 với target `"tack"`: bỏ từng chữ:

| bỏ | còn lại (tập) | có trong start? |
|---|---|---|
| t | {a, c, k} | không |
| a | {t, c, k} | không |
| c | {t, a, k} | không |
| k | {t, a, c} | **có** (`"act"`) ⇒ tạo được |

Target `"act"`: bỏ `a` → {c,t}, bỏ `c` → {a,t}, bỏ `t` → {a,c}: không có trong start ⇒ không tạo được (dù `"act"` có trong start — nhưng ta **phải** thêm một chữ).

Mỗi target chỉ có ≤ 26 cách bỏ ⇒ nếu tra cứu O(1) thì tổng O(26 · tổng độ dài).

In [6]:
start_sets = {frozenset(w) for w in examples[0][0]}
for t in examples[0][1]:
    rows = [(c, "".join(sorted(set(t) - {c})), frozenset(t) - {c} in start_sets) for c in t]
    print(f"target '{t}':", rows, "->", any(r[2] for r in rows))

target 'tack': [('t', 'ack', False), ('a', 'ckt', False), ('c', 'akt', False), ('k', 'act', True)] -> True
target 'act': [('a', 'ct', False), ('c', 'at', False), ('t', 'ac', False)] -> False
target 'acti': [('a', 'cit', False), ('c', 'ait', False), ('t', 'aci', False), ('i', 'act', True)] -> True


## 5. Cách 1 — Brute Force / baseline: so mọi cặp (start, target)

**Ý tưởng:** với mỗi target, duyệt mọi start, kiểm tra `len(t) == len(s) + 1` và `set(s) <= set(t)`.

In [7]:
def brute(startWords, targetWords):
    starts = [set(s) for s in startWords]
    ans = 0
    for t in targetWords:
        ts = set(t)
        if any(len(ts) == len(s) + 1 and s <= ts for s in starts):
            ans += 1
    return ans

for st, tg, exp in examples:
    got = brute(st, tg)
    print(got, "OK" if got == exp else "SAI")

2 OK
1 OK


**Độ phức tạp:** O(T · S · 26) với `S, T ≤ 5·10⁴` ⇒ ~6,5·10¹⁰ — TLE.

**Nút thắt:** với mỗi target ta xét mọi start, trong khi chỉ có **≤ 26** start "ứng viên" (mỗi ứng viên = target bỏ một chữ). Ta cần **tra cứu** thay vì **quét**.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** dựng một **tập băm** chứa "chữ ký" (signature) của mọi start — chữ ký là bất kỳ biểu diễn nào bất biến với hoán vị: chuỗi đã sắp xếp, `frozenset`, hoặc bitmask. Với mỗi target, thử bỏ từng chữ, tính chữ ký, tra tập băm.

Quy tắc:
- **Nếu** bỏ chữ `c` khỏi `t` cho chữ ký có trong tập **thì** đếm `t` và **dừng** (đừng đếm lại `t` với chữ khác).
- **Nếu** thử hết mọi chữ mà không có **thì** `t` không tạo được.

**Vì sao bitmask tốt nhất:** chữ ký là số nguyên ⇒ băm O(1) thật sự; bỏ một chữ chỉ là một phép XOR O(1) (thay vì tạo lại chuỗi O(26)). Tổng chi phí: O(Σ|start| + Σ|target|).

**Chiều thuận cũng được:** từ mỗi start thêm mỗi chữ **chưa có** ⇒ tập mọi mask "đạt được" (≤ 26·S phần tử), rồi đếm target có mask trong tập. Cùng độ phức tạp nhưng tốn bộ nhớ hơn (26 lần).

**Vì sao không cần lo "chữ thêm vào phải chưa có"?** Target không có chữ lặp, nên nếu `set(t) − {c}` = `set(s)` thì `c ∉ s` tự động.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Chuỗi đã sắp xếp làm chữ ký

In [8]:
def sorted_strings(startWords, targetWords):
    seen = {"".join(sorted(s)) for s in startWords}
    ans = 0
    for t in targetWords:
        ts = "".join(sorted(t))
        for i in range(len(ts)):
            if ts[:i] + ts[i + 1:] in seen:        # bỏ ký tự thứ i, vẫn còn sắp xếp
                ans += 1
                break
    return ans

for st, tg, exp in examples:
    got = sorted_strings(st, tg)
    print(got, "OK" if got == exp else "SAI")

2 OK
1 OK


Mẹo: sắp xếp target **một lần**; bỏ một ký tự khỏi chuỗi đã sắp xếp thì phần còn lại **vẫn** sắp xếp ⇒ khỏi sắp xếp lại 26 lần.

**Độ phức tạp cách 2:** O(S·L log L + T·L²) với `L ≤ 26` (mỗi lần cắt chuỗi O(L)).

### 7.2 Cách 3 — `frozenset` làm chữ ký

In [9]:
def frozensets(startWords, targetWords):
    seen = {frozenset(s) for s in startWords}
    ans = 0
    for t in targetWords:
        ft = frozenset(t)
        if any(ft - {c} in seen for c in t):
            ans += 1
    return ans

for st, tg, exp in examples:
    got = frozensets(st, tg)
    print(got, "OK" if got == exp else "SAI")

2 OK
1 OK


`any(...)` tự dừng ở lần trúng đầu tiên ⇒ mỗi target đếm tối đa một lần.

**Độ phức tạp cách 3:** O(T·L²) (mỗi `ft − {c}` tạo tập mới O(L) và băm O(L)).

### 7.3 Cách 4 — Bitmask, bỏ một chữ khỏi target (cách chuẩn)

In [10]:
def bitmask_remove(startWords, targetWords):
    seen = set()
    for s in startWords:
        m = 0
        for c in s:
            m |= 1 << (ord(c) - 97)
        seen.add(m)
    ans = 0
    for t in targetWords:
        m = 0
        for c in t:
            m |= 1 << (ord(c) - 97)
        for c in t:
            if (m ^ (1 << (ord(c) - 97))) in seen:
                ans += 1
                break
    return ans

for st, tg, exp in examples:
    got = bitmask_remove(st, tg)
    print(got, "OK" if got == exp else "SAI")

2 OK
1 OK


**Độ phức tạp cách 4:** O(Σ|s| + Σ|t|) — mỗi chữ của target: một XOR + một tra cứu số nguyên. Bộ nhớ O(S).

### 7.4 Cách 5 — Bitmask chiều thuận: thêm một chữ vào start

In [11]:
def bitmask_add(startWords, targetWords):
    reach = set()
    for s in startWords:
        m = to_mask(s)
        for b in range(26):
            if not (m >> b) & 1:
                reach.add(m | (1 << b))
    return sum(1 for t in targetWords if to_mask(t) in reach)

for st, tg, exp in examples:
    got = bitmask_add(st, tg)
    print(got, "OK" if got == exp else "SAI")

2 OK
1 OK


**Độ phức tạp cách 5:** O(26·S + Σ|t|) thời gian, O(26·S) bộ nhớ (~1,3 triệu số nguyên với S = 5·10⁴ — vẫn chấp nhận được). Ưu điểm: nếu có **nhiều** lượt hỏi target, tập `reach` dùng lại được.

## 8. TRACE từng bước

Trace cách 4 trên ví dụ 1.

In [12]:
def trace_bitmask(startWords, targetWords):
    seen = {}
    for s in startWords:
        seen[to_mask(s)] = s
        print(f"start '{s}': mask = {to_mask(s):>8} = {to_mask(s):026b}")
    ans = 0
    for t in targetWords:
        m = to_mask(t)
        print(f"target '{t}': mask = {m}")
        hit = None
        for c in t:
            r = m ^ (1 << (ord(c) - 97))
            ok = r in seen
            print(f"    bỏ '{c}' -> {r:>8} {'TRÚNG start ' + repr(seen[r]) if ok else 'không có'}")
            if ok: hit = c; break
        ans += hit is not None
        print(f"  -> {'đếm' if hit else 'không đếm'}, ans = {ans}")
    return ans

trace_bitmask(*examples[0][:2])

start 'ant': mask =   532481 = 00000010000010000000000001
start 'act': mask =   524293 = 00000010000000000000000101
start 'tack': mask =   525317 = 00000010000000010000000101
target 'tack': mask = 525317
    bỏ 't' ->     1029 không có
    bỏ 'a' ->   525316 không có
    bỏ 'c' ->   525313 không có
    bỏ 'k' ->   524293 TRÚNG start 'act'
  -> đếm, ans = 1
target 'act': mask = 524293
    bỏ 'a' ->   524292 không có
    bỏ 'c' ->   524289 không có
    bỏ 't' ->        5 không có
  -> không đếm, ans = 1
target 'acti': mask = 524549
    bỏ 'a' ->   524548 không có
    bỏ 'c' ->   524545 không có
    bỏ 't' ->      261 không có
    bỏ 'i' ->   524293 TRÚNG start 'act'
  -> đếm, ans = 2


2

Bảng tóm tắt:

| target | mask | bỏ chữ | kết quả |
|---|---|---|---|
| tack | {a,c,k,t} | t, a, c: không; **k** → {a,c,t} = "act" | đếm |
| act | {a,c,t} | a → {c,t}, c → {a,t}, t → {a,c}: không | không |
| acti | {a,c,i,t} | a, c: không; t → {a,c,i}: không; **i** → {a,c,t} | đếm |

Đáp án 2.

## 9. Code nộp LeetCode

In [13]:
from typing import List

class Solution:
    def wordCount(self, startWords: List[str], targetWords: List[str]) -> int:
        def mask(w: str) -> int:
            m = 0
            for c in w:
                m |= 1 << (ord(c) - 97)
            return m

        seen = {mask(s) for s in startWords}
        ans = 0
        for t in targetWords:
            m = mask(t)
            for c in t:
                if (m ^ (1 << (ord(c) - 97))) in seen:   # bỏ đúng một chữ
                    ans += 1
                    break                                 # mỗi target đếm một lần
        return ans

In [14]:
sol = Solution()
for st, tg, exp in examples:
    got = sol.wordCount(st, tg)
    print(got, "| kỳ vọng", exp, "| OK" if got == exp else "| SAI")

2 | kỳ vọng 2 | OK
1 | kỳ vọng 1 | OK


## 10. Vì sao thuật toán đúng

Ký hiệu `A(w)` = tập chữ của `w`. Vì không có chữ lặp, `mask(w)` là song ánh từ `A(w)` sang số nguyên và `|A(w)| = len(w)`.

**Bổ đề:** `t` tạo được từ `s` ⇔ ∃ `c ∈ A(t)`: `A(t) ∖ {c} = A(s)`.
- (⇒) Biến đổi thêm chữ `c ∉ A(s)` rồi hoán vị ⇒ `A(t) = A(s) ∪ {c}` ⇒ `A(t) ∖ {c} = A(s)` (vì `c ∉ A(s)`).
- (⇐) Đặt `c` là chữ bị bỏ; `c ∉ A(s)` (vì `A(s) = A(t) ∖ {c}`), nên thêm `c` vào `s` là hợp lệ, được tập `A(t)`; hoán vị để ra đúng `t`. ✔

Với bitmask: `A(t) ∖ {c}` ↔ `mask(t) ^ (1 << c)` (bit `c` đang bật nên XOR = tắt). Tra `in seen` ⇔ tồn tại start có đúng tập đó.

Thuật toán đếm `t` ⇔ vòng `for c in t` tìm thấy một `c` thỏa bổ đề, và `break` đảm bảo mỗi phần tử của `targetWords` (kể cả các phần tử trùng nhau) góp đúng 0 hoặc 1. ∎

## 11. Phân tích Time / Space Complexity

`S, T` = số chuỗi start/target; `L ≤ 26` độ dài tối đa.

| Cách | Time | Space |
|---|---|---|
| 1. Mọi cặp | O(S·T·L) | O(S·L) |
| 2. Chuỗi sắp xếp | O(S·L log L + T·L²) | O(S·L) |
| 3. frozenset | O(S·L + T·L²) | O(S·L) |
| 4. Bitmask bỏ chữ | O(S·L + T·L) | O(S) |
| 5. Bitmask thêm chữ | O(S·26 + T·L) | O(26·S) |

In [15]:
rng = random.Random(2135)
def rand_word(rng, lo=1, hi=26):
    return "".join(rng.sample(string.ascii_lowercase, rng.randint(lo, hi)))
S = [rand_word(rng, 1, 8) for _ in range(50000)]
T = [rand_word(rng, 2, 9) for _ in range(50000)]
res = {}
for name, f in [("sorted_strings", sorted_strings), ("frozensets", frozensets),
                ("bitmask_remove", bitmask_remove), ("bitmask_add", bitmask_add), ("Solution", Solution().wordCount)]:
    t0 = time.perf_counter(); res[name] = f(S, T); dt = time.perf_counter() - t0
    print(f"{name:<15} -> {res[name]}  {dt*1000:.0f} ms")
assert len(set(res.values())) == 1

sorted_strings  -> 28545  58 ms
frozensets      -> 28545  148 ms
bitmask_remove  -> 28545  53 ms


bitmask_add     -> 28545  260 ms
Solution        -> 28545  54 ms


## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [16]:
ALL = {"brute": brute, "sorted_strings": sorted_strings, "frozensets": frozensets,
       "bitmask_remove": bitmask_remove, "bitmask_add": bitmask_add, "Solution": Solution().wordCount}
edge = examples + [
    (["a"], ["a"], 0),                  # phải thêm đúng một chữ
    (["a"], ["ab", "ba", "ab"], 3),     # target trùng nhau: đếm riêng
    (["a"], ["abc"], 0),                # thêm 2 chữ: không được
    (["ab", "ac", "bc"], ["abc"], 1),   # nhiều cách nhưng đếm 1
    (["xyz"], ["zyxw", "wxyz", "xyzw"], 3),
    (["abcdefghijklmnopqrstuvwxy"], ["abcdefghijklmnopqrstuvwxyz"], 1),
    (["b", "c"], ["a", "z"], 0),       # target dài 1 không bao giờ được (start dài ≥ 1)
]
for st, tg, exp in edge:
    for name, f in ALL.items():
        assert f(list(st), list(tg)) == exp, (name, st, tg, exp)
print("Edge case OK:", len(edge))

Edge case OK: 9


### 12.2 Đối chiếu ngẫu nhiên với brute force (bảng chữ nhỏ để hay trúng)

In [17]:
rng = random.Random(42)
for _ in range(1500):
    alpha = string.ascii_lowercase[:rng.randint(2, 6)]
    def w():
        return "".join(rng.sample(alpha, rng.randint(1, len(alpha))))
    st = [w() for _ in range(rng.randint(1, 8))]
    tg = [w() for _ in range(rng.randint(1, 8))]
    ref = brute(st, tg)
    for name, f in ALL.items():
        assert f(st, tg) == ref, (name, st, tg)
print("1500 case ngẫu nhiên OK")

1500 case ngẫu nhiên OK


### 12.3 Tính chất: hoán vị chữ trong từng chuỗi không đổi đáp án

In [18]:
rng = random.Random(7)
for _ in range(300):
    st = [rand_word(rng, 1, 5) for _ in range(rng.randint(1, 20))]
    tg = [rand_word(rng, 2, 6) for _ in range(rng.randint(1, 20))]
    base = Solution().wordCount(st, tg)
    shuf = lambda ws: ["".join(rng.sample(x, len(x))) for x in ws]
    assert Solution().wordCount(shuf(st), shuf(tg)) == base
print("bất biến hoán vị: OK")

bất biến hoán vị: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Cho phép thêm **0** chữ (kiểm tra `⊆` thay vì "đúng một chữ")

In [19]:
def wrong_subset(startWords, targetWords):
    starts = [set(s) for s in startWords]
    return sum(1 for t in targetWords if any(s <= set(t) for s in starts))   # SAI: thiếu điều kiện độ dài

print("bản sai :", wrong_subset(*examples[0][:2]))
print("bản đúng:", Solution().wordCount(*examples[0][:2]))

bản sai : 3
bản đúng: 2


**Vì sao sai:** `"act" ⊆ "act"` nên bản sai đếm target `"act"`, và `"ab" ⊆ "abcd"` (thêm 2 chữ) cũng bị đếm. **Sửa:** thêm `len(t) == len(s) + 1`, hoặc dùng cách "bỏ đúng một chữ".

### Lỗi 2 — Quên `break`: một target đếm nhiều lần

In [20]:
def wrong_no_break(startWords, targetWords):
    seen = {to_mask(s) for s in startWords}; ans = 0
    for t in targetWords:
        m = to_mask(t)
        for c in t:
            if (m ^ (1 << (ord(c) - 97))) in seen:
                ans += 1                       # SAI: không break
    return ans

st, tg = ["ab", "ac", "bc"], ["abc"]
print("bản sai :", wrong_no_break(st, tg))
print("bản đúng:", Solution().wordCount(st, tg))

bản sai : 3
bản đúng: 1


**Sửa:** `break` sau khi đếm, hoặc dùng `any(...)`.

### Lỗi 3 — Khử trùng lặp `targetWords` bằng `set`

In [21]:
def wrong_dedupe_targets(startWords, targetWords):
    seen = {to_mask(s) for s in startWords}
    targets = set(targetWords)                       # SAI: target trùng chỉ còn 1
    return sum(1 for t in targets if any((to_mask(t) ^ (1 << (ord(c) - 97))) in seen for c in t))

st, tg = ["a"], ["ab", "ba", "ab"]
print("bản sai :", wrong_dedupe_targets(st, tg))
print("bản đúng:", Solution().wordCount(st, tg))

bản sai : 2
bản đúng: 3


**Vì sao sai:** đề hỏi "số chuỗi **trong** `targetWords`" — mỗi phần tử (kể cả trùng) được đếm. Khử trùng ở `startWords` thì **không sao** (chỉ cần tồn tại), nhưng ở `targetWords` thì sai. **Sửa:** duyệt `targetWords` gốc.

### Lỗi 4 — So sánh chuỗi gốc thay vì chữ ký (quên rằng được sắp xếp lại)

In [22]:
def wrong_no_sort(startWords, targetWords):
    seen = set(startWords); ans = 0
    for t in targetWords:
        if any(t[:i] + t[i + 1:] in seen for i in range(len(t))):   # SAI: "tack" bỏ 'k' = "tac" != "act"
            ans += 1
    return ans

print("bản sai :", wrong_no_sort(*examples[0][:2]))
print("bản đúng:", Solution().wordCount(*examples[0][:2]))

bản sai : 1
bản đúng: 2


**Sửa:** chuẩn hóa cả hai phía (sắp xếp / frozenset / bitmask) trước khi so sánh.

### Lỗi 5 — Kiểm tra bit bằng `m & (1 << b) == 1`

In [23]:
def wrong_bit_test(startWords, targetWords):
    reach = set()
    for s in startWords:
        m = to_mask(s)
        for b in range(26):
            if not (m & (1 << b) == 1):          # SAI: m & (1<<b) là 0 hoặc 1<<b, không phải 1
                reach.add(m | (1 << b))          # -> "thêm" cả chữ đã có (b > 0)
    return sum(1 for t in targetWords if to_mask(t) in reach)

st, tg = ["ab"], ["ab", "abc"]
print("bản sai :", wrong_bit_test(st, tg))
print("bản đúng:", Solution().wordCount(st, tg))

bản sai : 2
bản đúng: 1


**Vì sao sai:** với `b = 1` (chữ `'b'`, đang có trong `"ab"`), `m & 2 = 2 ≠ 1` nên bản sai tưởng `'b'` **chưa có** và "thêm" nó: `m | 2 = m` ⇒ `reach` chứa chính mask của `"ab"` ⇒ target `"ab"` bị đếm (thêm 0 chữ). **Sửa:** `if not (m >> b) & 1:` hoặc `if (m & (1 << b)) == 0:`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Sắp xếp lại tùy ý" / "anagram" ⇒ dùng **chữ ký bất biến hoán vị** (sorted, Counter, bitmask nếu không lặp chữ).
- "Thêm/bớt **đúng một** phần tử" ⇒ đảo chiều: từ phía lớn **bỏ từng phần tử** rồi tra cứu (≤ L lần).
- Tập con của bảng chữ cái 26 chữ, không lặp ⇒ **bitmask** nhanh và gọn.
- Luôn đóng ngoặc biểu thức bit khi so sánh.
- Bài cùng họ: 49 Group Anagrams, 1178 Number of Valid Words for Each Puzzle (bitmask + duyệt tập con), 318 Maximum Product of Word Lengths, 2157 Groups of Strings (thêm/bớt/thay một chữ + DSU).

## 15. Biến thể của bài

### 15.1 Biến thể A — Được thêm **đúng `k`** chữ

Bỏ `k` chữ khỏi target: C(L, k) tổ hợp — vẫn ổn với `k` nhỏ. Hoặc nhóm start theo độ dài và kiểm tra tập con bằng bitmask: `(ms & mt) == ms` và `popcount(mt) − popcount(ms) == k`.

In [24]:
from itertools import combinations
def word_count_k(startWords, targetWords, k):
    seen = {to_mask(s) for s in startWords}; ans = 0
    for t in targetWords:
        m = to_mask(t)
        if any((m ^ sum(1 << (ord(c) - 97) for c in comb)) in seen for comb in combinations(t, k)):
            ans += 1
    return ans

def word_count_k_brute(startWords, targetWords, k):
    return sum(1 for t in targetWords if any(len(t) == len(s) + k and set(s) <= set(t) for s in startWords))

print(word_count_k(["ab", "a"], ["abc", "abcd"], 2), word_count_k(["ab", "a"], ["abc", "abcd"], 1))
rng = random.Random(151)
for _ in range(500):
    alpha = "abcdef"
    st = ["".join(rng.sample(alpha, rng.randint(1, 4))) for _ in range(rng.randint(1, 6))]
    tg = ["".join(rng.sample(alpha, rng.randint(1, 6))) for _ in range(rng.randint(1, 6))]
    k = rng.randint(0, 3)
    assert word_count_k(st, tg, k) == word_count_k_brute(st, tg, k)
print("khớp brute force")

2 1
khớp brute force


### 15.2 Biến thể B — Trả về, với mỗi target, **danh sách start** dùng được

In [25]:
from collections import defaultdict
def sources(startWords, targetWords):
    by_mask = defaultdict(list)
    for s in startWords: by_mask[to_mask(s)].append(s)
    out = []
    for t in targetWords:
        m = to_mask(t); found = []
        for c in t:
            found += [(s, c) for s in by_mask.get(m ^ (1 << (ord(c) - 97)), [])]
        out.append(found)
    return out

for t, f in zip(["tack", "act", "acti"], sources(["ant", "act", "tack", "cat"], ["tack", "act", "acti"])):
    print(f"{t}: (start, chữ thêm) = {f}")

tack: (start, chữ thêm) = [('act', 'k'), ('cat', 'k')]
act: (start, chữ thêm) = []
acti: (start, chữ thêm) = [('act', 'i'), ('cat', 'i')]


### 15.3 Biến thể C — Start **có thể có chữ lặp** (multiset)

Bitmask không còn đúng (mất số lần). Dùng chữ ký là chuỗi đã sắp xếp (giữ số lần) và điều kiện "thêm một chữ chưa có" phải kiểm tra riêng: chữ bị bỏ khỏi target phải **không còn** trong phần còn lại.

In [26]:
def word_count_multiset(startWords, targetWords):
    seen = {"".join(sorted(s)) for s in startWords}; ans = 0
    for t in targetWords:
        ts = "".join(sorted(t))
        for i in range(len(ts)):
            rest = ts[:i] + ts[i + 1:]
            if ts[i] not in rest and rest in seen:     # chữ thêm phải chưa có trong start
                ans += 1; break
    return ans

print(word_count_multiset(["aab", "ab"], ["aabc", "aab", "abb"]))   # aabc <- aab + c ; abb: thêm b vào ab không hợp lệ

1


## 16. So sánh các cách làm

| Cách | Chữ ký | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi cặp | set | O(S·T·L) | O(S·L) | kiểm chứng |
| 2. Chuỗi sắp xếp | sorted str | O(S·L log L + T·L²) | O(S·L) | chuỗi có thể lặp chữ (biến thể C) |
| 3. frozenset | frozenset | O(S·L + T·L²) | O(S·L) | dễ đọc |
| 4. Bitmask bỏ chữ | int | O(S·L + T·L) | O(S) | **mặc định**, nhanh nhất |
| 5. Bitmask thêm chữ | int | O(26·S + T·L) | O(26·S) | nhiều lượt truy vấn target |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `wordCount(["g","vf","ylpuk","nyf","gdj","j","fyqzg","sizec"], ["r","am","jg","umhjo","fov","lujy","b","uz","y"])` là bao nhiêu? (Gợi ý: chỉ cần tìm target nào bỏ một chữ ra start.)

**Bài 2 (tính tay):** Tính `to_mask("bad")` dưới dạng nhị phân 4 bit cuối và thập phân. Bỏ chữ `'d'` thì được số nào?

**Bài 3 (điền TODO):** Viết `bt3_todo(startWords, targetWords)` trả về **danh sách các target KHÔNG tạo được** (giữ thứ tự, giữ trùng lặp).

**Bài 4 (tự viết lại):** Viết `bt4_todo(...)` cài cách 5 (chiều thuận: thêm chữ vào start).

**Bài 5 (mini challenge):** Viết `bt5_todo(startWords, targetWords)` đếm target tạo được nếu phép biến đổi là **thay** đúng một chữ của start bằng một chữ khác chưa có (thay vì thêm), rồi sắp xếp lại tùy ý.

In [27]:
def bt3_todo(startWords, targetWords):
    # TODO: các target không tạo được
    pass

def bt4_todo(startWords, targetWords):
    # TODO: bitmask chiều thuận
    pass

def bt5_todo(startWords, targetWords):
    # TODO: phép "thay một chữ"
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: target dài 1 không bao giờ được. Mức 2: `"jg"` bỏ `j` → `"g"` ✔; `"fov"` bỏ `o` → `{f, v}` = `"vf"` ✔; `"am"`, `"uz"`, `"lujy"`, `"umhjo"` bỏ chữ nào cũng không ra start. Mức 3: đáp án `2`.

**Bài 2.** Mức 1: a = bit 0, b = bit 1, d = bit 3. Mức 2: `1011₂`. Mức 3: `11`; bỏ `d` ⇒ `0011₂ = 3` = `to_mask("ab")`.

**Bài 3.** Mức 1: như `Solution` nhưng gom target thất bại. Mức 2: dùng `for … else`. Mức 3: `else: out.append(t)` sau vòng `for c in t` không `break`.

**Bài 4.** Mức 1: với mỗi start, thử 26 chữ. Mức 2: chỉ thêm chữ **chưa có**: `not (m >> b) & 1`. Mức 3: đếm target có `to_mask(t) in reach`.

**Bài 5.** Mức 1: thay một chữ ⇒ cùng độ dài và `|A(s) ∩ A(t)| = L − 1`. Mức 2: bỏ một chữ khỏi target **và** bỏ một chữ khỏi start cho cùng một tập. Mức 3: dựng bảng "lõi → tập chữ đã bị bỏ" từ mọi start (lõi = start bỏ một chữ). Target đếm được nếu bỏ chữ `c` của nó ra một lõi có trong bảng **và** tập chữ đã bị bỏ của lõi đó chứa một chữ **khác** `c` (nếu chỉ có `c` thì start trùng target — "thay `c` bằng `c`" không hợp lệ).

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `2` (`"jg"` từ `"g"`/`"j"`, `"fov"` từ `"vf"`). **Bài 2:** `to_mask("bad") = 0b1011 = 11`; bỏ `'d'` được `3`.

In [28]:
assert Solution().wordCount(["g", "vf", "ylpuk", "nyf", "gdj", "j", "fyqzg", "sizec"],
                            ["r", "am", "jg", "umhjo", "fov", "lujy", "b", "uz", "y"]) == 2
assert to_mask("bad") == 0b1011 == 11 and to_mask("bad") ^ (1 << 3) == 3 == to_mask("ab")

def bt3(startWords, targetWords):
    seen = {to_mask(s) for s in startWords}; out = []
    for t in targetWords:
        m = to_mask(t)
        for c in t:
            if (m ^ (1 << (ord(c) - 97))) in seen: break
        else:
            out.append(t)
    return out

def bt4(startWords, targetWords):
    reach = set()
    for s in startWords:
        m = to_mask(s)
        for b in range(26):
            if not (m >> b) & 1: reach.add(m | (1 << b))
    return sum(1 for t in targetWords if to_mask(t) in reach)

def bt5(startWords, targetWords):
    # với mỗi "lõi" (start bỏ một chữ) nhớ tập chữ đã bị bỏ để loại trường hợp thay chữ bằng chính nó
    core = defaultdict(set)
    for s in startWords:
        m = to_mask(s)
        for c in s:
            core[m ^ (1 << (ord(c) - 97))].add(c)
    ans = 0
    for t in targetWords:
        m = to_mask(t)
        for c in t:
            removed = core.get(m ^ (1 << (ord(c) - 97)))
            if removed and (removed - {c}):          # có start dùng chữ khác c ở vị trí đó
                ans += 1; break
    return ans

def bt5_brute(startWords, targetWords):
    return sum(1 for t in targetWords
               if any(len(s) == len(t) and len(set(s) & set(t)) == len(t) - 1 for s in startWords))

assert bt3(*examples[0][:2]) == ["act"] and bt3(["a"], ["a", "ab", "a"]) == ["a", "a"]
rng = random.Random(1935)
for _ in range(1500):
    alpha = string.ascii_lowercase[:rng.randint(2, 6)]
    st = ["".join(rng.sample(alpha, rng.randint(1, len(alpha)))) for _ in range(rng.randint(1, 7))]
    tg = ["".join(rng.sample(alpha, rng.randint(1, len(alpha)))) for _ in range(rng.randint(1, 7))]
    ref = brute(st, tg)
    assert bt4(st, tg) == ref
    assert len(bt3(st, tg)) == len(tg) - ref
    assert bt5(st, tg) == bt5_brute(st, tg)
print("Đáp án bài tập: tất cả assert OK")

Đáp án bài tập: tất cả assert OK


Ghi chú bài 5: nếu start `s` và target `t` có **cùng** tập chữ thì "bỏ chữ `c`" ở cả hai cho cùng lõi — nhưng đó là "thay `c` bằng `c`", không hợp lệ (chữ mới phải chưa có). Vì vậy ta lưu cho mỗi lõi **tập các chữ đã bị bỏ** và yêu cầu có một chữ **khác** `c`.

## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
def mask(w):
    m = 0
    for c in w: m |= 1 << (ord(c) - 97)
    return m
seen = {mask(s) for s in startWords}
ans = 0
for t in targetWords:
    m = mask(t)
    if any((m ^ (1 << (ord(c) - 97))) in seen for c in t):
        ans += 1
```

**Checklist:**
- [ ] Thêm **đúng một** chữ (không phải 0, không phải 2).
- [ ] Mỗi target đếm tối đa một lần (`break`/`any`).
- [ ] **Không** khử trùng `targetWords`.
- [ ] Chuẩn hóa thứ tự chữ (bitmask/sorted) ở **cả hai** phía.
- [ ] Đóng ngoặc biểu thức bit.